In [6]:
# my_live_demo.ipynb
# Демо объезда препятствий с обученной моделью

# ============================================
# 1. ИМПОРТЫ И ЗАГРУЗКА
# ============================================
%run my_camera.ipynb
%run my_basic.ipynb

print("🔧 Аппаратный сброс моторов...")
pcf.set_pin(0, 0)
pcf.set_pin(1, 0)
pcf.set_pin(2, 0)
pcf.set_pin(3, 0)
print("✅ Моторы сброшены в ноль")

import torch
import torch.nn as nn
from torchvision import transforms, models
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display
import threading
import time
from PIL import Image

# ============================================
# 2. НАСТРОЙКИ
# ============================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

# Путь к твоей обученной модели
MODEL_PATH = "/home/jetsonbot/jetbot/notebooks/my_teleop/my_model.pth"

# Параметры движения
FORWARD_SPEED = 0.3
TURN_SPEED = 0.2
PROB_THRESHOLD = 0.7  # порог вероятности препятствия

# ============================================
# 3. ЗАГРУЗКА МОДЕЛИ
# ============================================
print("🔍 Создаю архитектуру модели...")
model = models.resnet18(pretrained=False)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
print("✅ Архитектура создана")

print(f"📂 Загружаю веса из {MODEL_PATH}...")
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()
print("✅ Модель загружена и готова")

# ============================================
# 4. ЗАПУСК КАМЕРЫ
# ============================================
print("📷 Запуск камеры...")
camera = Camera()
camera.start()
time.sleep(2)
print("✅ Камера запущена")

# ============================================
# 5. ПРОВЕРКА ПЕРВОГО КАДРА
# ============================================
print("📸 Проверка получения кадра...")
test_frame = camera.read()
if test_frame is not None:
    print(f"✅ Кадр получен! Размер: {test_frame.shape}")
else:
    print("❌ Кадр НЕ получен!")

# ============================================
# 6. ВИДЖЕТ ДЛЯ ВИДЕО И СТАТУСА
# ============================================
video_widget = widgets.Image(format='jpeg', width=640, height=480)
prob_widget = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Вероятность blocked:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)
status_label = widgets.HTML(value="<b>Статус:</b> Ожидание...")

display(widgets.VBox([
    video_widget,
    prob_widget,
    status_label
]))
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================
# 7. ОСНОВНОЙ ЦИКЛ
# ============================================

def run():
    global running
    running = True
    print("🚀 Запуск автономного режима...")
    
    # Трансформации для нейросети
    local_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    frame_count = 0
    
    while running:  # <--- здесь отступ 4 пробела от начала функции
        frame = camera.read()  # <--- здесь отступ 8 пробелов от начала функции
        if frame is None:
            continue
            
        # Отображаем видео
        frame_resized = cv2.resize(frame, (640, 480))
        _, jpeg = cv2.imencode('.jpg', frame_resized, [cv2.IMWRITE_JPEG_QUALITY, 80])
        video_widget.value = jpeg.tobytes()
        
        frame_count += 1
        
        if frame_count % 3 == 0:
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(img)
            img_tensor = local_transform(img_pil).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = model(img_tensor)
                probabilities = torch.nn.functional.softmax(output[0], dim=0)
                blocked_prob = probabilities[1].item()
            
            prob_widget.value = blocked_prob
            
            if blocked_prob > PROB_THRESHOLD:
                status_label.value = f"<b>Статус:</b> ПРЕПЯТСТВИЕ! Поворот (p={blocked_prob:.3f})"
                left_motor.value = -TURN_SPEED
                right_motor.value = TURN_SPEED
            else:
                status_label.value = f"<b>Статус:</b> СВОБОДНО (p={blocked_prob:.3f})"
                left_motor.value = FORWARD_SPEED
                right_motor.value = FORWARD_SPEED
        
        time.sleep(0.03)
    
    print("🛑 Цикл завершён")

# ============================================
# 8. КНОПКИ УПРАВЛЕНИЯ
# ============================================
start_button = widgets.Button(
    description='🚀 СТАРТ',
    button_style='success',
    layout=widgets.Layout(width='150px', height='50px')
)

stop_button = widgets.Button(
    description='⏹ СТОП',
    button_style='danger',
    layout=widgets.Layout(width='150px', height='50px')
)

# Функции для кнопок
def on_start_clicked(b):
    global running_thread
    running_thread = threading.Thread(target=run)
    running_thread.daemon = True
    running_thread.start()

def on_stop_clicked(b):
    print("⏹️ АВАРИЙНАЯ ОСТАНОВКА (hard)")
    
    # Останавливаем бесконечный цикл
    global running
    running = False
    
    # Жёсткий сброс через PCF (все пины в 1 — выключено)
    pcf.output_state = 0xFF
    pcf._write()
    
    # Дополнительно через методы
    try:
        left_motor.hard_stop()
        right_motor.hard_stop()
    except:
        pass
    
    status_label.value = "<b>Статус:</b> АВАРИЙНЫЙ СТОП"
    prob_widget.value = 0.0
    print("✅ Моторы принудительно выключены")

start_button.on_click(on_start_clicked)
stop_button.on_click(on_stop_clicked)

display(widgets.HBox([start_button, stop_button]))
print("\n🎮 Нажми СТАРТ для запуска автономного режима")
print(f"   Порог препятствия: {PROB_THRESHOLD}")
print(f"   Скорость: {FORWARD_SPEED}, поворот: {TURN_SPEED}")

🔧 Аппаратный сброс моторов...
✅ Моторы сброшены в ноль
Используется устройство: cuda
🔍 Создаю архитектуру модели...
✅ Архитектура создана
📂 Загружаю веса из /home/jetsonbot/jetbot/notebooks/my_teleop/my_model.pth...
✅ Модель загружена и готова
📷 Запуск камеры...
Попытка открыть камеру с конвейером:
nvarguscamerasrc ! video/x-raw(memory:NVMM),width=640,height=480,framerate=30/1 ! nvvidconv ! video/x-raw,format=BGRx ! videoconvert ! video/x-raw,format=BGR ! appsink drop=1
✅ Камера успешно открыта через GStreamer.
📷 Поток захвата кадров запущен
✅ Камера запущена
📸 Проверка получения кадра...
✅ Кадр получен! Размер: (480, 640, 3)



🎮 Нажми СТАРТ для запуска автономного режима
   Порог препятствия: 0.7
   Скорость: 0.3, поворот: 0.2


In [ ]:
print(model)

In [2]:
# Аварийный стоп напрямую через PCF
pcf.set_pin(0, 0)  # левый мотор плюс
pcf.set_pin(1, 0)  # левый мотор минус
pcf.set_pin(3, 0)  # правый мотор плюс
pcf.set_pin(2, 0)  # правый мотор минус
print("Аварийный стоп выполнен")

Аварийный стоп выполнен
